# Fine-tuning rotary-IndicTrans2-200M on Marathi (en &rarr; mr)**Model:** `prajdabre/rotary-indictrans2-en-indic-dist-200M`**Data:** `ai4bharat/samanantar`, config `mr` (3.63M mined pairs)**Benchmark:** IN22-Gen (AI4Bharat), FLORES as fallback---### Why this notebook existsThe repo (`src/`) holds the real code; this notebook is the **driver** — it runsthe pipeline on a GPU in the order that catches mistakes earliest.The order is deliberate:| Step | What | Why here || ---: | --- | --- || 1 | Probe the tokenizer | IndicTrans2 has **two** SentencePiece vocabs. Using the source vocab for labels trains happily and learns nothing. This must be settled before anything else. || 2 | Overfit 32 pairs | Cheapest possible proof that gradients reach the target text. 2 minutes here saves a wasted 4-hour run. || 3 | Inspect the data funnel | Samanantar is *mined*, not curated. See what the filters actually remove. || 4 | **Baseline eval, before training** | You cannot claim an improvement without a number from before. Also a second check that decoding works. || 5 | The real run | Only now, once everything above is green. || 6-7 | Eval + qualitative diff | The deliverable. || 8 | Save to Drive | Colab wipes the disk on disconnect. |### Before you start`Runtime > Change runtime type > T4 GPU`. Free-tier T4 (16GB) is enough for theLoRA config in `configs/finetune_en_mr.yaml`.

---## 0. Check the GPUWhich GPU you get changes two things the code reads automatically:- **bf16 support.** T4 is Turing — fp16 only, *no bf16*. `src/modeling.py` picks  the right one instead of hardcoding, because asking a T4 for bf16 doesn't error,  it just runs badly.- **VRAM headroom.** 16GB comfortably fits LoRA at batch 16 / seq 128. If you see  <16GB, drop `training.per_device_train_batch_size` to 8.

In [ ]:
!nvidia-smi

import torch
print(f"torch          : {torch.__version__}")
print(f"cuda available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device         : {torch.cuda.get_device_name(0)}")
    print(f"vram           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bf16 supported : {torch.cuda.is_bf16_supported()}   <- False on T4, expected")

---## 1. Install dependenciesTwo things worth knowing:- **`transformers` is capped below 4.57.** IndicTrans2 ships its architecture as  *remote code* on the Hub, which is written against a specific transformers API.  Newest-is-best bites here; pin it.- **`IndicTransToolkit` is not on PyPI under that name** — install from GitHub. It  provides `IndicProcessor`, which does script normalisation and prepends the  `eng_Latn mar_Deva` language tags the model expects. Do **not** hand-roll this;  the tag format and the entity-masking are load-bearing.Colab may ask you to restart the runtime after this. If it does, restart andre-run from here — do *not* re-run cell 0.

In [ ]:
!pip install -q "transformers>=4.40,<4.57" "datasets>=2.18" accelerate sentencepiece
!pip install -q "sacrebleu>=2.4" "peft>=0.11" pyyaml pandas
!pip install -q git+https://github.com/VarunGumma/IndicTransToolkit.git

# Colab preinstalls torchao 0.10, but PEFT's LoRA dispatcher requires >0.16 and
# RAISES rather than skipping when the version is too old -- so every LoRA run
# dies at adapter injection. We use no quantization, so removing it is the
# cleanest fix: the availability check then returns False instead of throwing.
!pip uninstall -y -q torchao 2>/dev/null || true

import transformers, datasets, peft, sacrebleu
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("peft        ", peft.__version__)
print("sacrebleu   ", sacrebleu.__version__)

# Confirm the toolkit imports -- this is the install most likely to have failed.
from IndicTransToolkit.processor import IndicProcessor
print("IndicTransToolkit OK")

---## 2. Get the codeEither clone your GitHub repo (preferred — keeps notebook and repo in sync), orupload the folder to Colab. Everything below assumes the working directory is therepo root, so `python -m src.train` resolves.

In [ ]:
import os, subprocess, sys

REPO_URL  = "https://github.com/VVISHUS/AI_4_bharat_MT_FT.git"
REPO_NAME = "AI_4_bharat_MT_FT"
BRANCH    = "main"

# Always force the local checkout to match the remote. An earlier version of
# this cell only cloned when the directory was absent, which meant re-running it
# after a `git push` silently did NOTHING -- the stale code stayed, and the next
# cell failed with an error that had already been fixed upstream. A Colab
# "Restart session" keeps /content on disk, so restarting did not help either.
os.chdir("/content")
if not os.path.exists(REPO_NAME):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_NAME], check=True)

os.chdir(f"/content/{REPO_NAME}")
subprocess.run(["git", "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)

sys.path.insert(0, os.getcwd())   # so `from src.data import ...` works in-notebook

# Verify we actually have what we think we have, rather than trusting the pull.
head = subprocess.run(["git", "log", "--oneline", "-1"],
                      capture_output=True, text=True).stdout.strip()
train_src = open("src/train.py", encoding="utf-8").read()
print(f"cwd      : {os.getcwd()}")
print(f"HEAD     : {head}")
print(f"train.py : {train_src.count(chr(10)) + 1} lines")
print(f"collator : {'DataCollatorWithDecoderInputs' in train_src}")
assert "DataCollatorWithDecoderInputs" in train_src, (
    "Stale checkout -- the decoder_input_ids fix is missing. "
    "Delete /content/AI_4_bharat_MT_FT and re-run this cell."
)
print("
repo is in sync")

---## 3. STEP 1 — Probe the tokenizer**This is the single most important cell in the notebook.**IndicTrans2 uses **separate source and target SentencePiece vocabularies**. TheEnglish source and the Marathi target are *not* encoded by the same model.The failure this guards against is nasty because it is silent:> Encode Marathi labels with the English vocab &rarr; the text shatters into> character-level pieces and `<unk>` &rarr; training runs &rarr; loss decreases> smoothly &rarr; the model outputs garbage. You find out at evaluation, hours later.The probe prints which target-side API this tokenizer exposes (`text_target=`,a `src=` boolean, or `as_target_tokenizer()`) and does a **round-trip**:Marathi &rarr; ids &rarr; Marathi. A healthy Indic SPM uses well under 1 token percharacter; near 1.0 means character fallback, i.e. the wrong vocabulary.

In [ ]:
!python scripts/probe_tokenizer.py

**Read the output before continuing.**- `-> detected strategy: ...` — note which one. `src/data.py` uses the same  detection, so it will agree.- `tokens/char` should be roughly **0.2–0.5**. If it prints the "WRONG vocab"  warning, stop. Open the cached remote code  (`~/.cache/huggingface/modules/transformers_modules/`) and find where the  tokenizer selects the target SPM, then fix `encode_labels` in `src/data.py`.- The baseline translations in section 2 of the output should be readable  Marathi. If they are not, the problem is upstream of training entirely.

---## 4. STEP 2 — Overfit 32 pairsA model with ~200M parameters should be able to **memorise** 32 sentence pairsalmost completely. So: if loss does not collapse toward zero, something in thelabel pipeline is broken — masking, shifting, the vocab, the collator.This is a 2-minute test that de-risks the entire run. `--smoke` sets 30 epochson 32 examples, no eval, logging every 5 steps.**Pass condition:** final training loss **< 1.0** (the script enforces this andexits non-zero otherwise).

In [ ]:
!python -m src.train --config configs/finetune_en_mr.yaml --smoke

If loss plateaus around 4–6 and refuses to move, the usual suspects, in order oflikelihood:1. **Labels encoded with the source vocab** — go back to step 1.2. **`decoder_start_token_id` is None** — the model can't begin generating.   Check what the probe printed.3. **Label padding not masked to `-100`** — the model spends its capacity   learning to predict padding. `DataCollatorForSeq2Seq(label_pad_token_id=-100)`   handles this; confirm the collator is actually being used.4. **LR far too low.** Unlikely at 1e-4, but try 5e-4 in smoke mode to isolate.

**If it crashes at `attach_lora` with a torchao ImportError**, the install cell's
`pip uninstall torchao` did not take. Either re-run it, or skip LoRA entirely:

```
!python -m src.train --config configs/finetune_en_mr.yaml --smoke --set peft.enabled=false
```

Full fine-tuning fits a T4 at this model size (~3.2GB for weights + grads +
Adam state, against 16GB available). LoRA buys headroom here, not feasibility.

---## 5. STEP 3 — Look at the data funnelSamanantar is **bitext-mined**: sentence pairs were automatically aligned fromcomparable corpora, so misalignment, duplication and wrong-language rows are allexpected at some rate.The filters in `configs/finetune_en_mr.yaml` each target a specific failure:| Filter | Catches || --- | --- || `length_bounds` | fragments, headers, runaway rows that blow up padding || `length_ratio` | misaligned pairs (one side is not a translation of the other) || `script_check` | "Marathi" rows that are actually English, URLs, or number tables || `copy_pairs` | src == tgt, i.e. the miner gave up || `dedup_source` | Samanantar has many targets per source; keep one |Run this cell to see the funnel *and read some surviving pairs yourself*. Eyeballing30 rows of your training data is worth more than any amount of config tuning.

In [ ]:
import pandas as pd
from src.config import load_config
from src.data import load_samanantar_pairs

cfg = load_config("configs/finetune_en_mr.yaml")

# Small pool for a quick look; the real run uses the full configured size.
peek_cfg = {**cfg["data"], "max_train_samples": 3000, "valid_samples": 200}
pairs, report = load_samanantar_pairs(peek_cfg)

print(report.to_markdown())
print(f"\nkept {report.kept} of {report.raw} rows streamed\n")

pd.set_option("display.max_colwidth", 90)
display(pd.DataFrame(pairs[:30]))

Scan that table. You are looking for:- Pairs where the Marathi is clearly **not** a translation of the English  &rarr; tighten `max_word_ratio`.- Marathi rows carrying lots of Latin text &rarr; raise `min_devanagari_ratio`.- Near-duplicate sources with trivial variations &rarr; a fuzzy dedup would help,  though exact dedup is usually enough at this scale.Whatever you find, **write it in the README with the funnel table.** Thefiltering decisions and their justification are the most visible evidence ofjudgement in this project.

---## 6. STEP 4 — Baseline evaluation, *before* trainingMeasure first. Three reasons:1. You cannot report a delta without a "before".2. It's an end-to-end test of the *inference* path (generate &rarr; decode &rarr;   postprocess) while you still have time to fix it.3. IndicTrans2 is already strong at en&rarr;mr. Knowing the starting chrF++ sets   honest expectations: a 120k-pair LoRA run on in-domain mined data may well   **not** beat it, and that is a perfectly reportable result.Benchmark is **IN22-Gen** (AI4Bharat's own), not held-out Samanantar — scoring onheld-out rows of a mined corpus measures how well you fit the miner's noise.`src/evaluate.py` falls back to FLORES automatically if the Hub config name has drifted.

In [ ]:
!python -m src.evaluate --config configs/finetune_en_mr.yaml --base-only --output outputs/baseline

---## 7. STEP 5 — The real fine-tuning runEverything is green, so now we spend the GPU hours.**Configuration rationale** (all in `configs/finetune_en_mr.yaml`):| Choice | Value | Why || --- | --- | --- || LoRA | r=16, &alpha;=32 | ~1% trainable params. Optimiser state stays tiny, so batch 16 fits a T4 with headroom. Full FT also fits — set `peft.enabled=false` to compare. || LR | 1e-4 | LoRA tolerates ~10&times; the full-FT LR since only adapters move. || Effective batch | 32 | 16 &times; 2 grad-accum. MT likes larger batches; this is the most we get cheaply. || Label smoothing | 0.1 | Standard for NMT — reduces overconfidence, usually worth ~0.5 chrF. || `group_by_length` | true | Batches similar-length sequences, cutting padding waste substantially on variable-length MT data. || Scheduler | cosine + 3% warmup | Warmup matters: a cold high LR on a pretrained MT model degrades it fast. || Epochs | 1 | On 120k pairs that's ~3.7k optimiser steps. Enough to move the model; short enough to finish and still have time to write the README. |**Colab will disconnect if you close the tab.** Keep it open. Checkpoints landevery 500 steps, so a disconnect costs you at most that.

In [ ]:
# ~45-90 min on a T4 at these settings. Watch the first 100 steps:
# loss should fall steadily. If it spikes and stays high, kill it and halve the LR.
!python -m src.train --config configs/finetune_en_mr.yaml

---## 8. STEP 6 — Evaluate the fine-tuned modelSame benchmark, same generation settings (beams, max length) as the baseline, sothe comparison isolates the weights rather than the decoding.`src/evaluate.py` detects a LoRA checkpoint by the presence of`adapter_config.json`, reloads the base, applies the adapter, and merges it forfaster decoding.

In [ ]:
!python -m src.evaluate \
    --config configs/finetune_en_mr.yaml \
    --checkpoint outputs/rotary-it2-en-mr/final \
    --output outputs/rotary-it2-en-mr

### Reading the result honestly**chrF++ is the headline.** BLEU with a single reference on a morphologically richtarget is noisy and rewards surface n-gram overlap; report it because it'sexpected, not because it's informative here.**If fine-tuned < base:** that is a genuine, publishable result, not a failure.Say so and give the mechanism. The most likely one here: IndicTrans2 was trainedon BPCC, a superset that already includes Samanantar, *plus* far more curateddata. Fine-tuning on 120k mined pairs narrows the distribution and can undo someof that. Supporting evidence to cite: the gap is larger on IN22-Gen (out ofdomain) than on Samanantar validation (in domain).A candid analysis of a regression reads far better to this panel than a luckynumber. Don't tune until it goes green — explain it.

---## 9. STEP 7 — Qualitative diffMetrics hide things. Read the actual outputs.Look specifically at: named entities and numbers (does postprocessing restorethem?), sentence-final verb agreement, and whether the fine-tuned model hasstarted truncating or repeating — the classic symptom of over-fitting a short-sentence corpus.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("outputs/rotary-it2-en-mr/qualitative_samples.md", encoding="utf-8").read()))

---## 10. STEP 8 — Save artifacts to DriveColab deletes the disk on disconnect. Save before you celebrate.What to keep (all small — the LoRA adapter is a few MB, not 800):- `final/` — the adapter (or full checkpoint)- `resolved_config.json` — exact config including CLI overrides- `filter_report.md` / `.json` — the data funnel- `train_metrics.json`, `eval_results.json`, `qualitative_samples.md`- `trainer_state.json` — the full loss curve, for a plot in the README

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil, os
DEST = "/content/drive/MyDrive/ai4bharat_mt_ft"
os.makedirs(DEST, exist_ok=True)

for run in ["outputs/rotary-it2-en-mr", "outputs/baseline"]:
    if os.path.exists(run):
        target = os.path.join(DEST, os.path.basename(run))
        shutil.copytree(run, target, dirs_exist_ok=True)
        print("saved ->", target)

!du -sh $DEST/*

---## 11. Plot the loss curve for the README`trainer_state.json` holds every logged step. A loss curve is the fastest way fora reviewer to see that the run actually trained.

In [ ]:
import json
import matplotlib.pyplot as plt

state = json.load(open("outputs/rotary-it2-en-mr/final/trainer_state.json", encoding="utf-8"))
history = state["log_history"]

train = [(h["step"], h["loss"]) for h in history if "loss" in h]
evals = [(h["step"], h["eval_chrf++"]) for h in history if "eval_chrf++" in h]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(*zip(*train))
axes[0].set(xlabel="step", ylabel="training loss", title="Training loss")
axes[0].grid(alpha=0.3)

if evals:
    axes[1].plot(*zip(*evals), marker="o", color="tab:green")
    axes[1].set(xlabel="step", ylabel="chrF++", title="Validation chrF++ (in-domain)")
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/rotary-it2-en-mr/loss_curve.png", dpi=140)
plt.show()

---## What goes in the READMEYou now have everything the writeup needs. The parts that earn credit:1. **The document-level mismatch.** This checkpoint's card says the rotary   variants are *"primarily built and tested for document-level and long-context   translations."* Samanantar is sentence-level. Name that mismatch, say you   proceeded anyway and why, and describe what you'd do with a document-aligned   corpus (BPCC-doc) instead. Noticing this is worth more than any metric.2. **The two-vocabulary trap** and how you tested for it rather than assuming.3. **The filter funnel table**, with the reasoning per filter.4. **The honest metric delta**, including a regression if that's what happened,   with a mechanism rather than a shrug.5. **What you'd do with a week**: full BPCC rather than a 120k subset, full   fine-tuning compared against LoRA, document-level training to match the   checkpoint's design, and human evaluation on a sample — chrF++ is a proxy.